In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [ ]:
class LayerNorm(nn.Module):
    def __init__(self, normalized_shape, eps=1e-05, elementwise_affine=True, bias=True):
        super().__init__()
        self.normalized_shape = normalized_shape if isinstance(normalized_shape, (tuple, list)) else (normalized_shape,) # Shape of tensor excluding batch
        self.eps = eps # Avoid 0 division
        self.elementwise_affine = elementwise_affine # If true, this module has learnable per-element affine parameters
        self.bias = bias # If False, the layer will not learn an additive bias

        if self.elementwise_affine:
            self.scale = nn.Parameter(torch.ones(self.normalized_shape)) # Initiate to one for no-op
            if self.bias:
                self.shift = nn.Parameter(torch.zeros(self.normalized_shape)) # Initiate to zero for no-op
            else:
                self.register_parameter("shift", None) # Register parameter as None

    def forward(self, x):
        dims = [-(i+1) for i in range(len(self.normalized_shape))] # Get the list of dimensions
        mean = torch.mean(x, dim=dims, keepdim=True) # # Compute the mean for each layer
        var = x.var(dim=dims, unbiased=False, keepdim=True) # Compute the var for each layer, correction=0 for population variance
        norm = (x - mean) / torch.sqrt(var + self.eps)

        if self.elementwise_affine:
            norm = self.scale * norm
            if self.shift is not None:
                norm += self.shift

        return norm

In [ ]:
x = torch.randn(64, 3, 32, 32)
bn = LayerNorm((3,32,32))
bn.train()

out = bn(x) # Forward pass

print(out.mean(dim=(0, 2, 3)))
print(out.std(dim=(0, 2, 3)))

tensor([ 0.0016, -0.0078,  0.0061], grad_fn=<MeanBackward1>)
tensor([1.0011, 1.0021, 0.9967], grad_fn=<StdBackward0>)
